In [4]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import re

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np


In [3]:

# List of required files for a valid cluster
required_files = [
    "test_files.txt",
    "tprateAutoPYara_Test.txt",
    "tprateAutoPYara_Train.txt",
    "tprateAutoyaraBase_Test.txt",
    "tprateAutoyaraBase_Train.txt",
    "train_files.txt"
]

# The tprate files we're interested in
tprate_files = {
    "AutoPYara_Test": "tprateAutoPYara_Test.txt",
    "AutoPYara_Train": "tprateAutoPYara_Train.txt",
    "AutoyaraBase_Test": "tprateAutoyaraBase_Test.txt",
    "AutoyaraBase_Train": "tprateAutoyaraBase_Train.txt"
}

def parse_tp_rate(line):
    # Parse line like "TP Rate: 0.4737 (9/19)"
    match = re.search(r"TP Rate: (\d+\.\d+) \((\d+)/(\d+)\)", line.strip())
    if match:
        tp_rate = float(match.group(1))
        tp = int(match.group(2))
        total = int(match.group(3))
        return tp_rate, tp, total
    return None, None, None

def check_and_extract(folder_path):
    # Get list of cluster folders
    cluster_folders = [f for f in os.listdir(folder_path) if os.path.isdir(os.path.join(folder_path, f)) and f.startswith("cluster_")]
    
    # Initialize lists for each dataframe
    data = {key: [] for key in tprate_files.keys()}
    
    # Check each cluster folder
    for cluster in cluster_folders:
        cluster_path = os.path.join(folder_path, cluster)
        # Check if all required files exist
        all_files_present = all(os.path.isfile(os.path.join(cluster_path, file)) for file in required_files)
        
        if all_files_present:
            # Extract data from each tprate file
            for df_name, file_name in tprate_files.items():
                file_path = os.path.join(cluster_path, file_name)
                try:
                    with open(file_path, 'r') as f:
                        content = f.read().strip()
                        tp_rate, tp, total = parse_tp_rate(content)
                        if tp_rate is not None:
                            data[df_name].append({
                                'cluster': cluster,
                                'tp_rate': tp_rate,
                                'tp': tp,
                                'total': total
                            })
                except Exception as e:
                    print(f"Error reading {file_path}: {e}")
                    continue
    
    # Create dataframes
    dataframes = [pd.DataFrame(data[name]) for name in tprate_files.keys()]
    
    return dataframes

In [3]:
# folder_path = ['/home/mabon/research/Autoyara/YaraResults/yaraRules/retrainedBloomFilters/ssdeep/BuildHeuristics/StreamClusterData/Th50/Ratio_0.25/',
#                '/home/mabon/research/Autoyara/YaraResults/yaraRules/retrainedBloomFilters/ssdeep/BuildHeuristics/StreamClusterData/Th60/Ratio_0.25/',
#                '/home/mabon/research/Autoyara/YaraResults/yaraRules/retrainedBloomFilters/ssdeep/BuildHeuristics/StreamClusterData/Th70/Ratio_0.25/',
#                '/home/mabon/research/Autoyara/YaraResults/yaraRules/retrainedBloomFilters/ssdeep/BuildHeuristics/StreamClusterData/Th80/Ratio_0.25/',
#                '/home/mabon/research/Autoyara/YaraResults/yaraRules/retrainedBloomFilters/ssdeep/BuildHeuristics/StreamClusterData/Th90/Ratio_0.25/']

# AutoPYara_Test=[]
# AutoPYara_Train=[]
# AutoyaraBase_Test=[]
# AutoyaraBase_Train=[]

# for path in folder_path:
#     # Get dataframes
#     dfs = check_and_extract(path)
#     AutoPYara_Test.append(dfs[0])
#     AutoPYara_Train.append(dfs[1])
#     AutoyaraBase_Test.append(dfs[2])
#     AutoyaraBase_Train.append(dfs[3])
# import numpy as np
# import matplotlib.pyplot as plt
# import pandas as pd

# # Global font settings
# plt.rcParams['font.family'] = 'DejaVu Sans'
# plt.rcParams['font.weight'] = 600
# plt.rcParams['font.size'] = 22

# # Font dictionaries
# TITLE_FONT = {'family': 'DejaVu Sans', 'weight': 600, 'size': 22}
# AXIS_FONT = {'family': 'DejaVu Sans', 'weight': 900, 'size': 14.0}
# TICK_FONT = {'fontsize': 22, 'fontweight': 'bold'}

# def plot_cluster_std_dev_boxplot_merged_subplots(
#     auto_pyara_train, autoyara_base_train,
#     drop_smallest_n=0,
#     output_file='cumulative_median_train_subplots.pdf'
# ):
#     # Validate inputs
#     for dfs, name in [
#         (auto_pyara_train, 'AutoPYara_Train'),
#         (autoyara_base_train, 'AutoyaraBase_Train')
#     ]:
#         if not all(isinstance(df, pd.DataFrame) for df in dfs):
#             raise ValueError(f"All items in {name} must be pandas DataFrames")
#         if len(dfs) != 5:
#             raise ValueError(f"Expected exactly 5 DataFrames in {name}")
#     if not isinstance(drop_smallest_n, int) or drop_smallest_n < 0:
#         raise ValueError("drop_smallest_n must be a non-negative integer")

#     # Merge the five DataFrames for each dataset
#     auto_pyara_train_merged = pd.concat(auto_pyara_train, ignore_index=True)
#     autoyara_base_train_merged = pd.concat(autoyara_base_train, ignore_index=True)

#     datasets = [
#         ('AutoPYara Train', auto_pyara_train_merged, '#ff7f0e'),  # Orange
#         ('AutoyaraBase Train', autoyara_base_train_merged, '#d62728')  # Red
#     ]

#     # Collect all unique cluster sizes across all datasets
#     all_sizes = set()
#     for dataset_name, df, _ in datasets:
#         df = df.replace('None', np.nan).apply(pd.to_numeric, errors='coerce')
#         try:
#             sizes = df['total'].dropna().astype(int).tolist()
#             all_sizes.update(sizes)
#         except (ValueError, KeyError):
#             continue

#     all_sizes = sorted(all_sizes)
#     if drop_smallest_n > 0 and len(all_sizes) > drop_smallest_n:
#         all_sizes = all_sizes[drop_smallest_n:]
#     if not all_sizes:
#         raise ValueError("No valid cluster sizes found")

#     # Define custom ticks
#     tick_sizes = [0, 25, 50, 75, 100, 500]
#     tick_positions = []
#     for tick in tick_sizes:
#         if all_sizes:
#             closest_idx = min(range(len(all_sizes)), key=lambda i: abs(all_sizes[i] - tick))
#             tick_positions.append(closest_idx + 1)

#     # Create figure with single subplot for Train datasets
#     fig, ax = plt.subplots(1, 1, figsize=(15, 6))

#     # Plot for Train datasets
#     handles_train = []
#     labels_train = []
#     for dataset_name, df, color in datasets:
#         df = df.replace('None', np.nan).apply(pd.to_numeric, errors='coerce')
#         datasets_per_threshold = [
#             (df[df['total'] == size]['tp_rate'].dropna().astype(float) * 100).tolist()
#             if len(df[df['total'] == size]) > 0 else [0]
#             for size in all_sizes
#         ]
#         medians = [np.median(d) if d else np.nan for d in datasets_per_threshold]
#         valid_medians = [m for m in medians if not np.isnan(m) and m > 0]  # Exclude zeros
#         valid_positions = [j + 1 for j, d in enumerate(datasets_per_threshold) if d and np.median(d) > 0]

#         if valid_medians:
#             # Start cumulative calculation only after first non-zero median
#             cumulative_avg = np.cumsum(valid_medians) / np.arange(1, len(valid_medians) + 1)
#             line, = ax.plot(valid_positions[:len(cumulative_avg)], cumulative_avg,
#                            linewidth=2, color=color)
#             avg_value = np.mean(cumulative_avg)
#             ax.axhline(y=avg_value, color=color, linestyle=':', linewidth=1.5)
#             handles_train.append(line)
#             labels_train.append(dataset_name)

#     ax.set_xticks(tick_positions)
#     ax.set_xticklabels([str(t) for t in tick_sizes], **TICK_FONT)
#     ax.set_xlabel('Cluster Size (Number of Items)', **TICK_FONT)
#     ax.set_ylabel('True Positive (%)', **TICK_FONT)
#     ax.set_ylim(0, 100)
#     ax.set_title('Train Datasets', **TITLE_FONT)
#     ax.legend(
#         handles_train, labels_train,
#         loc='upper center',
#         bbox_to_anchor=(0.5, 1.02),
#         ncol=2,
#         prop=AXIS_FONT,
#         frameon=False
#     )
#     ax.grid(True, linestyle='--', alpha=0.5)

#     # Adjust layout and save
#     fig.tight_layout()
#     fig.savefig(output_file, format='pdf', bbox_inches='tight')
#     plt.close(fig)

# # Example usage (replace with actual DataFrames)
# plot_cluster_std_dev_boxplot_merged_subplots(
#     AutoPYara_Train,
#     AutoyaraBase_Train,
#     drop_smallest_n=0,
#     output_file='25.pdf'
# )

In [95]:
folder_path = ['/home/mabon/research/Autoyara/YaraResults/yaraRules/retrainedBloomFilters/sdhash/AutoPYara/StreamClusterData_HEU/Th50/Ratio_0.25/',
               '/home/mabon/research/Autoyara/YaraResults/yaraRules/retrainedBloomFilters/sdhash/AutoPYara/StreamClusterData_HEU/Th60/Ratio_0.25/',
               '/home/mabon/research/Autoyara/YaraResults/yaraRules/retrainedBloomFilters/sdhash/AutoPYara/StreamClusterData_HEU/Th70/Ratio_0.25/',
               '/home/mabon/research/Autoyara/YaraResults/yaraRules/retrainedBloomFilters/sdhash/AutoPYara/StreamClusterData_HEU/Th80/Ratio_0.25/',
               '/home/mabon/research/Autoyara/YaraResults/yaraRules/retrainedBloomFilters/sdhash/AutoPYara/StreamClusterData_HEU/Th90/Ratio_0.25/']

autopyar_s1=[]
AutoPYara_Train_s1=[]
autoyar_s1=[]
AutoyaraBase_Train_s1=[]

for path in folder_path:
    # Get dataframes
    dfs = check_and_extract(path)
    autopyar_s1.append(dfs[0])
    AutoPYara_Train_s1.append(dfs[1])
    autoyar_s1.append(dfs[2])
    AutoyaraBase_Train_s1.append(dfs[3])

folder_path = ['/home/mabon/research/Autoyara/YaraResults/yaraRules/retrainedBloomFilters/sdhash/AutoPYara/StreamClusterData_HEU/Th50/Ratio_0.5/',
               '/home/mabon/research/Autoyara/YaraResults/yaraRules/retrainedBloomFilters/sdhash/AutoPYara/StreamClusterData_HEU/Th60/Ratio_0.5/',
               '/home/mabon/research/Autoyara/YaraResults/yaraRules/retrainedBloomFilters/sdhash/AutoPYara/StreamClusterData_HEU/Th70/Ratio_0.5/',
               '/home/mabon/research/Autoyara/YaraResults/yaraRules/retrainedBloomFilters/sdhash/AutoPYara/StreamClusterData_HEU/Th80/Ratio_0.5/',
               '/home/mabon/research/Autoyara/YaraResults/yaraRules/retrainedBloomFilters/sdhash/AutoPYara/StreamClusterData_HEU/Th90/Ratio_0.5/']

autopyar_s2=[]
AutoPYara_Train_s2=[]
autoyar_s2=[]
AutoyaraBase_Train_s2=[]

for path in folder_path:
    # Get dataframes
    dfs = check_and_extract(path)
    autopyar_s2.append(dfs[0])
    AutoPYara_Train_s2.append(dfs[1])
    autoyar_s2.append(dfs[2])
    AutoyaraBase_Train_s2.append(dfs[3])

folder_path = ['/home/mabon/research/Autoyara/YaraResults/yaraRules/retrainedBloomFilters/sdhash/AutoPYara/StreamClusterData_HEU/Th50/Ratio_0.75/',
               '/home/mabon/research/Autoyara/YaraResults/yaraRules/retrainedBloomFilters/sdhash/AutoPYara/StreamClusterData_HEU/Th60/Ratio_0.75/',
               '/home/mabon/research/Autoyara/YaraResults/yaraRules/retrainedBloomFilters/sdhash/AutoPYara/StreamClusterData_HEU/Th70/Ratio_0.75/',
               '/home/mabon/research/Autoyara/YaraResults/yaraRules/retrainedBloomFilters/sdhash/AutoPYara/StreamClusterData_HEU/Th80/Ratio_0.75/',
               '/home/mabon/research/Autoyara/YaraResults/yaraRules/retrainedBloomFilters/sdhash/AutoPYara/StreamClusterData_HEU/Th90/Ratio_0.75/']

autopyar_s3=[]
AutoPYara_Train_s3=[]
autoyar_s3=[]
AutoyaraBase_Train_s3=[]
for path in folder_path:
    # Get dataframes
    dfs = check_and_extract(path)
    autopyar_s3.append(dfs[0])
    AutoPYara_Train_s3.append(dfs[1])
    autoyar_s3.append(dfs[2])
    AutoyaraBase_Train_s3.append(dfs[3])

       

In [6]:
# import numpy as np
# import matplotlib.pyplot as plt
# import pandas as pd

# # Global font settings
# plt.rcParams['font.family'] = 'DejaVu Sans'
# plt.rcParams['font.weight'] = 600
# plt.rcParams['font.size'] = 22

# # Font dictionaries
# TITLE_FONT = {'family': 'DejaVu Sans', 'weight': 600, 'size': 22}
# AXIS_FONT = {'family': 'DejaVu Sans', 'weight': 900, 'size': 14.0}
# TICK_FONT = {'fontsize': 22, 'fontweight': 'bold'}

# def plot_cluster_std_dev_boxplot_merged_subplots(
#     auto_pyara_train, autoyara_base_train,
#     output_file='cumulative_median_train_subplots.pdf'
# ):
#     # Validate inputs
#     for dfs, name in [
#         (auto_pyara_train, 'AutoPYara_Train'),
#         (autoyara_base_train, 'AutoyaraBase_Train')
#     ]:
#         if not all(isinstance(df, pd.DataFrame) for df in dfs):
#             raise ValueError(f"All items in {name} must be pandas DataFrames")
#         if len(dfs) != 5:
#             raise ValueError(f"Expected exactly 5 DataFrames in {name}")

#     # Merge the five DataFrames for each dataset
#     auto_pyara_train_merged = pd.concat(auto_pyara_train, ignore_index=True)
#     autoyara_base_train_merged = pd.concat(autoyara_base_train, ignore_index=True)

#     datasets = [
#         ('AutoPYara Train', auto_pyara_train_merged, '#ff7f0e'),  # Orange
#         ('AutoyaraBase Train', autoyara_base_train_merged, '#d62728')  # Red
#     ]

#     # Collect all unique cluster sizes across all datasets
#     all_sizes = set()
#     for dataset_name, df, _ in datasets:
#         df = df.replace('None', np.nan).apply(pd.to_numeric, errors='coerce')
#         try:
#             sizes = df['total'].dropna().astype(int).tolist()
#             all_sizes.update(sizes)
#         except (ValueError, KeyError):
#             continue

#     all_sizes = sorted(all_sizes)
#     if not all_sizes:
#         raise ValueError("No valid cluster sizes found")

#     # Automatically determine the minimum cluster size to start the plot
#     min_cluster_size = min(all_sizes)  # Smallest cluster size in the data

#     # Filter all_sizes to start from min_cluster_size
#     all_sizes = [size for size in all_sizes if size >= min_cluster_size]

#     # Define custom ticks, starting from min_cluster_size
#     tick_sizes = [min_cluster_size, 25, 50, 75, 100, 500]
#     tick_sizes = [t for t in tick_sizes if t >= min_cluster_size]  # Keep only relevant ticks
#     tick_positions = []
#     for tick in tick_sizes:
#         if all_sizes:
#             closest_idx = min(range(len(all_sizes)), key=lambda i: abs(all_sizes[i] - tick))
#             tick_positions.append(closest_idx + 1)

#     # Create figure with single subplot for Train datasets
#     fig, ax = plt.subplots(1, 1, figsize=(15, 6))

#     # Plot for Train datasets
#     handles_train = []
#     labels_train = []
#     for dataset_name, df, color in datasets:
#         df = df.replace('None', np.nan).apply(pd.to_numeric, errors='coerce')
#         datasets_per_threshold = [
#             (df[df['total'] == size]['tp_rate'].dropna().astype(float) * 100).tolist()
#             if len(df[df['total'] == size]) > 0 else [0]
#             for size in all_sizes
#         ]
#         medians = [np.median(d) if d else np.nan for d in datasets_per_threshold]
#         valid_medians = [m for m in medians if not np.isnan(m) and m > 0]  # Exclude zeros
#         valid_positions = [j + 1 for j, d in enumerate(datasets_per_threshold) if d and np.median(d) > 0]

#         if valid_medians:
#             # Start cumulative calculation only after first non-zero median
#             cumulative_avg = np.cumsum(valid_medians) / np.arange(1, len(valid_medians) + 1)
#             line, = ax.plot(valid_positions[:len(cumulative_avg)], cumulative_avg,
#                            linewidth=2, color=color)
#             avg_value = np.mean(cumulative_avg)
#             ax.axhline(y=avg_value, color=color, linestyle=':', linewidth=1.5)
#             handles_train.append(line)
#             labels_train.append(dataset_name)

#     ax.set_xticks(tick_positions)
#     ax.set_xticklabels([str(t) for t in tick_sizes], **TICK_FONT)
#     ax.set_xlabel('Cluster Size (Number of Items)', **TICK_FONT)
#     ax.set_ylabel('True Positive (%)', **TICK_FONT)
#     ax.set_ylim(0, 100)
#     ax.set_title('Train Datasets', **TITLE_FONT)
#     ax.legend(
#         handles_train, labels_train,
#         loc='upper center',
#         bbox_to_anchor=(0.5, 1.02),
#         ncol=2,
#         prop=AXIS_FONT,
#         frameon=False
#     )
#     ax.grid(True, linestyle='--', alpha=0.5)

#     # Adjust layout and save
#     fig.tight_layout()
#     fig.savefig(output_file, format='pdf', bbox_inches='tight')
#     plt.close(fig)

# # Example usage (replace with actual DataFrames)
# plot_cluster_std_dev_boxplot_merged_subplots(
#     AutoPYara_Train_s2,
#     AutoyaraBase_Train_s2,
#     output_file='25.pdf'
# )

In [28]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Global font settings
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.weight'] = 600
plt.rcParams['font.size'] = 22

# Font dictionaries
TITLE_FONT = {'family': 'DejaVu Sans', 'weight': 600, 'size': 22}
AXIS_FONT = {'family': 'DejaVu Sans', 'weight': 900, 'size': 18.0}
TICK_FONT = {'fontsize': 22, 'fontweight': 'bold'}

def plot_cluster_std_dev_boxplot_merged_subplots(
    auto_pyara_train_s1, autoyara_base_train_s1,
    auto_pyara_train_s2, autoyara_base_train_s2,
    auto_pyara_train_s3, autoyara_base_train_s3,
    output_file='cumulative_median_train_subplots.pdf'
):

    # Define datasets with names and colors
    datasets = [
        ('AutoPYara 0.25', pd.concat(auto_pyara_train_s1, ignore_index=True), '#ff7f0e'),  # Orange
        ('AutoYara 0.25', pd.concat(autoyara_base_train_s1, ignore_index=True), '#d62728'),  # Red
        ('AutoPYara 0.50', pd.concat(auto_pyara_train_s2, ignore_index=True), '#1f77b4'),  # Blue
        ('AutoYara 0.50', pd.concat(autoyara_base_train_s2, ignore_index=True), '#2ca02c'),  # Green
        ('AutoPYara 0.75', pd.concat(auto_pyara_train_s3, ignore_index=True), '#9467bd'),  # Purple
        ('AutoYara 0.75', pd.concat(autoyara_base_train_s3, ignore_index=True), '#8c564b')  # Brown
    ]
    
    # Collect all unique cluster sizes across all datasets
    all_sizes = set()
    min_sizes = {}  # Store minimum cluster size per dataset
    for dataset_name, df, _ in datasets:
        df = df.replace('None', np.nan).apply(pd.to_numeric, errors='coerce')
        try:
            sizes = df['total'].dropna().astype(int).tolist()
            if sizes:
                all_sizes.update(sizes)
                min_sizes[dataset_name] = min(sizes)
        except (ValueError, KeyError):
            continue

    all_sizes = sorted(all_sizes)
    if not all_sizes:
        raise ValueError("No valid cluster sizes found")

    # Determine global minimum cluster size for x-axis
    global_min_cluster_size = min(all_sizes)

    # Filter all_sizes to start from global_min_cluster_size
    all_sizes = [size for size in all_sizes if size >= global_min_cluster_size]

    # Define custom ticks, starting from global_min_cluster_size
    tick_sizes = [global_min_cluster_size, 25, 50, 75, 100, 1000]
    tick_sizes = [t for t in tick_sizes if t >= global_min_cluster_size]
    tick_positions = []
    for tick in tick_sizes:
        if all_sizes:
            closest_idx = min(range(len(all_sizes)), key=lambda i: abs(all_sizes[i] - tick))
            tick_positions.append(closest_idx + 1)

    # Create figure with single subplot
    fig, ax = plt.subplots(1, 1, figsize=(15, 6))

    # Plot for each dataset
    handles_train = []
    labels_train = []
    for dataset_name, df, color in datasets:
        df = df.replace('None', np.nan).apply(pd.to_numeric, errors='coerce')
        # Start from the dataset's minimum cluster size
        dataset_min_size = min_sizes.get(dataset_name, global_min_cluster_size)
        # Filter sizes to include only those >= dataset_min_size
        dataset_sizes = [size for size in all_sizes if size >= dataset_min_size]
        if not dataset_sizes:
            continue
        datasets_per_threshold = [
            (df[df['total'] == size]['tp_rate'].dropna().astype(float) * 100).tolist()
            if len(df[df['total'] == size]) > 0 else [0]
            for size in all_sizes
        ]
        medians = [np.median(d) if d else np.nan for d in datasets_per_threshold]
        valid_medians = [m for m in medians if not np.isnan(m) and m > 0]
        valid_positions = [j + 1 for j, d in enumerate(datasets_per_threshold) if d and np.median(d) > 0]
        
        # Adjust positions to align with dataset_min_size
        dataset_min_index = all_sizes.index(dataset_min_size) + 1 if dataset_min_size in all_sizes else 1
        valid_positions = [p for p in valid_positions if p >= dataset_min_index]
        valid_medians = [medians[p-1] for p in valid_positions]

        if valid_medians:
            cumulative_avg = np.cumsum(valid_medians) / np.arange(1, len(valid_medians) + 1)
            line, = ax.plot(valid_positions[:len(cumulative_avg)], cumulative_avg,
                           linewidth=2, color=color)
            avg_value = np.mean(cumulative_avg)
            ax.axhline(y=avg_value, color=color, linestyle=':', linewidth=1.5)
            handles_train.append(line)
            labels_train.append(dataset_name)

    ax.set_xticks(tick_positions)
    ax.set_xticklabels([str(t) for t in tick_sizes], **TICK_FONT)
    ax.set_xlabel('Cluster Size (Number of Items)', **TICK_FONT)
    ax.set_ylabel('True Positive (%)', **TICK_FONT)
    ax.set_ylim(0, 100)
    # ax.set_title('Threat Hunting', **TITLE_FONT)
    ax.legend(
        handles_train, labels_train,
        loc='lower center',
        bbox_to_anchor=(0.5, .1),
        ncol=3,
        prop=AXIS_FONT,
        frameon=False
    )
    ax.grid(True, linestyle='--', alpha=0.5)

    # Adjust layout and save
    fig.tight_layout()
    fig.savefig(output_file, format='pdf', bbox_inches='tight')
    plt.close(fig)


In [29]:

# Example usage (replace with actual DataFrames)
plot_cluster_std_dev_boxplot_merged_subplots(
    AutoPYara_Train_s1,
    AutoyaraBase_Train_s1,
    AutoPYara_Train_s2,
    AutoyaraBase_Train_s2,
    AutoPYara_Train_s3,
    AutoyaraBase_Train_s3,
    output_file='train_sdhash.pdf'
)

In [31]:

# Example usage (replace with actual DataFrames)
plot_cluster_std_dev_boxplot_merged_subplots(
    autopyar_s1,
    autoyar_s1,
    autopyar_s2,
    autoyar_s2,
    autopyar_s3,
    autoyar_s3,
    output_file='test_sdhash.pdf'
)

In [74]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Global font settings
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.weight'] = 900
plt.rcParams['font.size'] = 22

# Font dictionaries
TITLE_FONT = {'family': 'DejaVu Sans', 'weight': 900, 'size': 22}
AXIS_FONT = {'family': 'DejaVu Sans', 'weight': 900, 'size': 18.0}
TICK_FONT = {'fontsize': 22, 'fontweight': 'bold'}

def plot_cluster_std_dev_mirrored(
    auto_pyara_train_s1, autoyara_base_train_s1,
    auto_pyara_train_s2, autoyara_base_train_s2,
    auto_pyara_train_s3, autoyara_base_train_s3,
    auto_pyara_test_s1, autoyara_base_test_s1,
    auto_pyara_test_s2, autoyara_base_test_s2,
    auto_pyara_test_s3, autoyara_base_test_s3,
    output_file='mirrored_sdhash.pdf'
):
    # Define datasets for training and test
    datasets = [
        ('AutoPYara 0.25', pd.concat(auto_pyara_train_s1, ignore_index=True), '#ff7f0e', 'train'),  # Orange
        ('AutoYara 0.25', pd.concat(autoyara_base_train_s1, ignore_index=True), '#d62728', 'train'),  # Red
        ('AutoPYara 0.50', pd.concat(auto_pyara_train_s2, ignore_index=True), '#1f77b4', 'train'),  # Blue
        ('AutoYara 0.50', pd.concat(autoyara_base_train_s2, ignore_index=True), '#2ca02c', 'train'),  # Green
        ('AutoPYara 0.75', pd.concat(auto_pyara_train_s3, ignore_index=True), '#9467bd', 'train'),  # Purple
        ('AutoYara 0.75', pd.concat(autoyara_base_train_s3, ignore_index=True), '#8c564b', 'train'),  # Brown
        ('AutoPYara 0.25', pd.concat(auto_pyara_test_s1, ignore_index=True), '#ff7f0e', 'test'),   # Orange
        ('AutoYara 0.25', pd.concat(autoyara_base_test_s1, ignore_index=True), '#d62728', 'test'),   # Red
        ('AutoPYara 0.50', pd.concat(auto_pyara_test_s2, ignore_index=True), '#1f77b4', 'test'),   # Blue
        ('AutoYara 0.50', pd.concat(autoyara_base_test_s2, ignore_index=True), '#2ca02c', 'test'),   # Green
        ('AutoPYara 0.75', pd.concat(auto_pyara_test_s3, ignore_index=True), '#9467bd', 'test'),   # Purple
        ('AutoYara 0.75', pd.concat(autoyara_base_test_s3, ignore_index=True), '#8c564b', 'test')    # Brown
    ]

    # Collect all unique cluster sizes across all datasets
    all_sizes = set()
    min_sizes = {}  # Store minimum cluster size per dataset
    for dataset_name, df, _, data_type in datasets:
        df = df.replace('None', np.nan).apply(pd.to_numeric, errors='coerce')
        try:
            sizes = df['total'].dropna().astype(int).tolist()
            if sizes:
                all_sizes.update(sizes)
                min_sizes[dataset_name + '_' + data_type] = min(sizes)
        except (ValueError, KeyError):
            continue

    all_sizes = sorted(all_sizes)
    if not all_sizes:
        raise ValueError("No valid cluster sizes found")

    # Determine global minimum cluster size for x-axis (e.g., 2)
    global_min_cluster_size = min(all_sizes)

    # Filter all_sizes to start from global_min_cluster_size
    all_sizes = [size for size in all_sizes if size >= global_min_cluster_size]

    # Define custom ticks, starting from global_min_cluster_size (e.g., 2)
    tick_sizes = [global_min_cluster_size, 25, 50, 75, 100, 500]
    tick_sizes = [t for t in tick_sizes if t >= global_min_cluster_size]
    tick_positions = []
    for tick in tick_sizes:
        if all_sizes:
            closest_idx = min(range(len(all_sizes)), key=lambda i: abs(all_sizes[i] - tick))
            tick_positions.append(closest_idx + 1)

    # Adjust tick positions to place first tick at x=0
    first_tick_pos = tick_positions[0]  # Position for global_min_cluster_size (e.g., 2)
    adjusted_positions = [0] + [p - first_tick_pos for p in tick_positions[1:]]  # Center first tick at 0
    adjusted_sizes = tick_sizes  # Keep same sizes for labels

    # Create figure and single subplot
    fig, ax = plt.subplots(figsize=(20, 8))
    
    # Create twin x-axis for test data (for top labels)
    ax_test = ax.twiny()

    # Combined handles and labels for legend (unique by name and color)
    handles = []
    labels = []
    plotted_labels = set()

    # Plot for all datasets
    for dataset_name, df, color, data_type in datasets:
        df = df.replace('None', np.nan).apply(pd.to_numeric, errors='coerce')
        dataset_min_size = min_sizes.get(dataset_name + '_' + data_type, global_min_cluster_size)
        dataset_sizes = [size for size in all_sizes if size >= dataset_min_size]
        if not dataset_sizes:
            continue
        datasets_per_threshold = [
            (df[df['total'] == size]['tp_rate'].dropna().astype(float) * 100).tolist()
            if len(df[df['total'] == size]) > 0 else [0]
            for size in all_sizes
        ]
        medians = [np.median(d) if d else np.nan for d in datasets_per_threshold]
        valid_medians = [m for m in medians if not np.isnan(m) and m > 0]
        valid_positions = [j + 1 for j, d in enumerate(datasets_per_threshold) if d and np.median(d) > 0]
        dataset_min_index = all_sizes.index(dataset_min_size) + 1 if dataset_min_size in all_sizes else 1
        valid_positions = [p for p in valid_positions if p >= dataset_min_index]
        valid_medians = [medians[p-1] for p in valid_positions]

        if valid_medians:
            # Adjust positions to center first tick at 0
            plot_positions = [-(p - first_tick_pos) for p in valid_positions] if data_type == 'train' else [p - first_tick_pos for p in valid_positions]
            cumulative_avg = np.cumsum(valid_medians) / np.arange(1, len(valid_medians) + 1)
            line, = ax.plot(plot_positions[:len(cumulative_avg)], cumulative_avg,
                           linewidth=2, color=color)
            avg_value = np.mean(cumulative_avg)
            ax.axhline(y=avg_value, color=color, linestyle=':', linewidth=1.5)
            # Add to legend only if not already added
            if dataset_name not in plotted_labels:
                handles.append(line)
                labels.append(dataset_name)
                plotted_labels.add(dataset_name)

    # Move y-axis to leftmost edge
    ax.spines['left'].set_position(('axes', 0))  # Left edge of plot
    ax.spines['right'].set_color('none')
    ax.spines['bottom'].set_position('zero')
    ax.spines['top'].set_color('none')

    # Set symmetric x-axis limits
    max_pos = max([abs(p) for p in adjusted_positions]) + 1
    ax.set_xlim(-max_pos, max_pos)
    ax_test.set_xlim(max_pos, -max_pos)  # Reverse for mirroring

    # Set x-axis ticks and labels, with first tick (e.g., 2) at x=0
    ax.set_xticks([-p for p in adjusted_positions[1:]] + [0] + adjusted_positions[1:])
    ax.set_xticklabels([str(s) for s in adjusted_sizes[1:]] + [str(adjusted_sizes[0])] + [str(s) for s in adjusted_sizes[1:]], **TICK_FONT)
    ax_test.set_xticks([])  # Hide test x-axis ticks

    # Label axes
    ax.set_ylabel('True Positive (%)', **TICK_FONT)
    ax.set_xlabel('Train <--             Cluster Size            Test -->', **TICK_FONT, x=0.5)

    # Set y-axis limits
    ax.set_ylim(0, 100)

    # Add single legend above the plot
    ax.legend(
        handles, labels,
        loc='upper center',
        bbox_to_anchor=(0.5, 1.15),
        ncol=3,
        prop={'size': 18, 'family': 'DejaVu Sans', 'weight': 'bold'},
        frameon=False
    )

    # Add grid
    ax.grid(True, linestyle='--', alpha=0.5)

    # Adjust layout and save
    fig.subplots_adjust(top=0.85, left=0.15)  # Make room for legend and y-axis labels
    fig.savefig(output_file, format='pdf', bbox_inches='tight')
    plt.close(fig)

# Example usage
plot_cluster_std_dev_mirrored(
    AutoPYara_Train_s1, AutoyaraBase_Train_s1,
    AutoPYara_Train_s2, AutoyaraBase_Train_s2,
    AutoPYara_Train_s3, AutoyaraBase_Train_s3,
    autopyar_s1, autoyar_s1,
    autopyar_s2, autoyar_s2,
    autopyar_s3, autoyar_s3,
    output_file='mirrored_sdhash.pdf'
)

In [96]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Global font settings
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['font.size'] = 18

# Font dictionaries
TITLE_FONT = {'family': 'DejaVu Sans', 'weight': 'bold', 'size': 20}
AXIS_FONT = {'family': 'DejaVu Sans', 'weight': 'bold', 'size': 16}
TICK_FONT = {'fontsize': 16, 'fontweight': 'bold'}
LEGEND_FONT = {'size': 14, 'family': 'DejaVu Sans', 'weight': 'bold'}

def plot_cluster_std_dev_stacked(
    auto_pyara_train_s1, autoyara_base_train_s1,
    auto_pyara_train_s2, autoyara_base_train_s2,
    auto_pyara_train_s3, autoyara_base_train_s3,
    auto_pyara_test_s1, autoyara_base_test_s1,
    auto_pyara_test_s2, autoyara_base_test_s2,
    auto_pyara_test_s3, autoyara_base_test_s3,
    output_file='stacked_sdhash.pdf'
):
    # Define datasets grouped by threshold
    thresholds = [
        {
            'name': 'Threshold 0.25',
            'train': [
                ('AutoPYara', pd.concat(auto_pyara_train_s1, ignore_index=True), '#ff7f0e'),  # Orange
                ('AutoYara', pd.concat(autoyara_base_train_s1, ignore_index=True), '#d62728')   # Red
            ],
            'test': [
                ('AutoPYara', pd.concat(auto_pyara_test_s1, ignore_index=True), '#ff7f0e'),   # Orange
                ('AutoYara', pd.concat(autoyara_base_test_s1, ignore_index=True), '#d62728')   # Red
            ]
        },
        {
            'name': 'Threshold 0.50',
            'train': [
                ('AutoPYara', pd.concat(auto_pyara_train_s2, ignore_index=True), '#1f77b4'),  # Blue
                ('AutoYara', pd.concat(autoyara_base_train_s2, ignore_index=True), '#2ca02c')  # Green
            ],
            'test': [
                ('AutoPYara', pd.concat(auto_pyara_test_s2, ignore_index=True), '#1f77b4'),  # Blue
                ('AutoYara', pd.concat(autoyara_base_test_s2, ignore_index=True), '#2ca02c')  # Green
            ]
        },
        {
            'name': 'Threshold 0.75',
            'train': [
                ('AutoPYara', pd.concat(auto_pyara_train_s3, ignore_index=True), '#9467bd'),  # Purple
                ('AutoYara', pd.concat(autoyara_base_train_s3, ignore_index=True), '#8c564b')  # Brown
            ],
            'test': [
                ('AutoPYara', pd.concat(auto_pyara_test_s3, ignore_index=True), '#9467bd'),  # Purple
                ('AutoYara', pd.concat(autoyara_base_test_s3, ignore_index=True), '#8c564b')  # Brown
            ]
        }
    ]

    # Create figure with three subplots
    fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(20, 12), sharex=True, sharey=True)

    # Collect all unique cluster sizes across all datasets
    all_sizes = set()
    min_sizes = {}
    for thresh in thresholds:
        for dataset_name, df, _ in thresh['train'] + thresh['test']:
            df = df.replace('None', np.nan).apply(pd.to_numeric, errors='coerce')
            try:
                sizes = df['total'].dropna().astype(int).tolist()
                if sizes:
                    all_sizes.update(sizes)
                    min_sizes[f"{dataset_name}_{thresh['name']}"] = min(sizes)
            except (ValueError, KeyError):
                continue

    all_sizes = sorted(all_sizes)
    if not all_sizes:
        raise ValueError("No valid cluster sizes found")

    # Determine global minimum cluster size
    global_min_cluster_size = min(all_sizes)
    all_sizes = [size for size in all_sizes if size >= global_min_cluster_size]

    # Define custom ticks
    tick_sizes = [global_min_cluster_size, 25, 50, 75, 100, 1000]
    tick_sizes = [t for t in tick_sizes if t >= global_min_cluster_size]
    tick_positions = []
    for tick in tick_sizes:
        if all_sizes:
            closest_idx = min(range(len(all_sizes)), key=lambda i: abs(all_sizes[i] - tick))
            tick_positions.append(closest_idx)

    # Adjust tick positions to start at 0
    first_tick_pos = tick_positions[0]
    adjusted_positions = [p - first_tick_pos for p in tick_positions]
    adjusted_sizes = tick_sizes

    for idx, thresh in enumerate(thresholds):
        ax = axes[idx]
        ax_test = ax.twiny()  # Twin x-axis for test data

        handles = []
        labels = []
        plotted_labels = set()  

        # Plot train and test data
        for data_type, datasets in [('train', thresh['train']), ('test', thresh['test'])]:
            for dataset_name, df, color in datasets:
                df = df.replace('None', np.nan).apply(pd.to_numeric, errors='coerce')
                dataset_min_size = min_sizes.get(f"{dataset_name}_{thresh['name']}", global_min_cluster_size)
                dataset_sizes = [size for size in all_sizes if size >= dataset_min_size]
                if not dataset_sizes:
                    continue
                datasets_per_threshold = [
                    (df[df['total'] == size]['tp_rate'].dropna().astype(float) * 100).tolist()
                    if len(df[df['total'] == size]) > 0 else [0]
                    for size in all_sizes
                ]
                medians = [np.median(d) if d else np.nan for d in datasets_per_threshold]
                valid_medians = [m for m in medians if not np.isnan(m) and m > 0]
                valid_positions = [j for j, d in enumerate(datasets_per_threshold) if d and np.median(d) > 0]
                dataset_min_index = all_sizes.index(dataset_min_size) if dataset_min_size in all_sizes else 0
                valid_positions = [p for p in valid_positions if p >= dataset_min_index]
                valid_medians = [medians[p] for p in valid_positions]

                if valid_medians:
                    plot_positions = [-(p - first_tick_pos) for p in valid_positions] if data_type == 'train' else [p - first_tick_pos for p in valid_positions]
                    cumulative_avg = np.cumsum(valid_medians) / np.arange(1, len(valid_medians) + 1)
                    if dataset_name not in plotted_labels:
                        line, = ax.plot(plot_positions[:len(cumulative_avg)], cumulative_avg,
                                       linewidth=2, color=color, label=dataset_name)
                        handles.append(line)
                        labels.append(dataset_name)
                        plotted_labels.add(dataset_name)
                    else:
                        ax.plot(plot_positions[:len(cumulative_avg)], cumulative_avg,
                                linewidth=2, color=color)
                    avg_value = np.mean(cumulative_avg)
                    ax.axhline(y=avg_value, color=color, linestyle=':', linewidth=1.5)

        # Customize subplot
        ax.spines['left'].set_position(('axes', 0))
        ax.spines['right'].set_color('none')
        ax.spines['bottom'].set_position(('data', 50))
        ax.spines['top'].set_color('none')

        # Set x-axis ticks and labels
        ax.set_xticks([-p for p in adjusted_positions[1:]] + [0] + adjusted_positions[1:])
        ax.set_xticklabels([str(s) for s in adjusted_sizes[1:]] + [str(adjusted_sizes[0])] + [str(s) for s in adjusted_sizes[1:]], **TICK_FONT)

        ax_test.set_xticks([])  # Hide test x-axis ticks
        ax_test.set_xlim(max(adjusted_positions) + 1, -max(adjusted_positions) - 1)

        # Set y-axis label for middle subplot
        if idx == 1:
            ax.set_ylabel('True Positive (%)', **AXIS_FONT)
        ax.set_ylim(50, 100)

        # Add grid
        ax.grid(True, linestyle='--', alpha=0.5)

        # Set x-axis label
        ax.set_xlabel('Train <-- Cluster Size --> Test', **AXIS_FONT)

        # Add title below x-axis tick labels
        ax.text(0.5, -0.30, thresh['name'], transform=ax.transAxes, ha='center', va='top', **TITLE_FONT)

        # Add legend inside the plot (upper right)
        ax.legend(handles, labels, loc='lower left', prop=LEGEND_FONT, frameon=False)

    # Add overall title at the bottom of the figure

    # Adjust layout to accommodate bottom title
    fig.tight_layout(rect=[0.02, 0.02, 0.95, 0.95])  # Adjusted bottom margin for overall title

    # Save figure
    fig.savefig(output_file, format='pdf', bbox_inches='tight')
    plt.close(fig)
    # Adjust layout to accom
# Example usage
plot_cluster_std_dev_stacked(
    AutoPYara_Train_s1, AutoyaraBase_Train_s1,
    AutoPYara_Train_s2, AutoyaraBase_Train_s2,
    AutoPYara_Train_s3, AutoyaraBase_Train_s3,
    autopyar_s1, autoyar_s1,
    autopyar_s2, autoyar_s2,
    autopyar_s3, autoyar_s3,
    output_file='stacked_sdhash.pdf'
)

In [97]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Global font settings
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.weight'] = 900
plt.rcParams['font.size'] = 22

# Font dictionaries
TITLE_FONT = {'family': 'DejaVu Sans', 'weight': 900, 'size': 22}
AXIS_FONT = {'family': 'DejaVu Sans', 'weight': 900, 'size': 22.0}
TICK_FONT = {'fontsize': 22, 'fontweight': 'bold'}

def plot_cluster_std_dev_mirrored(
    auto_pyara_train_s3, autoyara_base_train_s3,
    auto_pyara_test_s3, autoyara_base_test_s3,
    output_file='mirrored_sdhash_0.75.pdf'
):
    """
    Plot mirrored cumulative average true positive rates for AutoPYara and AutoYara
    at threshold 0.75 for training and test datasets.
    
    Parameters:
    - auto_pyara_train_s3: List of DataFrames for AutoPYara train data (threshold 0.75)
    - autoyara_base_train_s3: List of DataFrames for AutoYara train data (threshold 0.75)
    - auto_pyara_test_s3: List of DataFrames for AutoPYara test data (threshold 0.75)
    - autoyara_base_test_s3: List of DataFrames for AutoYara test data (threshold 0.75)
    - output_file: File path to save the plot (default: 'mirrored_sdhash_0.75.pdf')
    """
    # Define datasets for training and test (only threshold 0.75)
    datasets = [
        ('AutoPYara 0.75', pd.concat(auto_pyara_train_s3, ignore_index=True), '#9467bd', 'train'),  # Purple
        ('AutoYara 0.75', pd.concat(autoyara_base_train_s3, ignore_index=True), '#8c564b', 'train'),  # Brown
        ('AutoPYara 0.75', pd.concat(auto_pyara_test_s3, ignore_index=True), '#9467bd', 'test'),   # Purple
        ('AutoYara 0.75', pd.concat(autoyara_base_test_s3, ignore_index=True), '#8c564b', 'test')    # Brown
    ]

    # Collect all unique cluster sizes across all datasets
    all_sizes = set()
    min_sizes = {}  # Store minimum cluster size per dataset
    for dataset_name, df, _, data_type in datasets:
        df = df.replace('None', np.nan).apply(pd.to_numeric, errors='coerce')
        try:
            sizes = df['total'].dropna().astype(int).tolist()
            if sizes:
                all_sizes.update(sizes)
                min_sizes[dataset_name + '_' + data_type] = min(sizes)
        except (ValueError, KeyError) as e:
            print(f"Error processing {dataset_name} ({data_type}): {e}")
            continue

    if not all_sizes:
        raise ValueError("No valid cluster sizes found in the provided datasets")

    # Sort cluster sizes
    all_sizes = sorted(all_sizes)
    global_min_cluster_size = min(all_sizes)
    all_sizes = [size for size in all_sizes if size >= global_min_cluster_size]

    # Define custom ticks, starting from global_min_cluster_size
    tick_sizes = [global_min_cluster_size, 25, 50, 75, 100, 1000]
    tick_sizes = [t for t in tick_sizes if t >= global_min_cluster_size]
    tick_positions = []
    for tick in tick_sizes:
        if all_sizes:
            closest_idx = min(range(len(all_sizes)), key=lambda i: abs(all_sizes[i] - tick))
            tick_positions.append(closest_idx + 1)

    # Adjust tick positions to place first tick at x=0
    first_tick_pos = tick_positions[0]
    adjusted_positions = [0] + [p - first_tick_pos for p in tick_positions[1:]]
    adjusted_sizes = tick_sizes

    # Create figure and single subplot
    fig, ax = plt.subplots(figsize=(16, 6))
    ax_test = ax.twiny()  # Twin x-axis for test data

    # Combined handles and labels for legend
    handles = []
    labels = []
    plotted_labels = set()

    # Plot for all datasets
    for dataset_name, df, color, data_type in datasets:
        df = df.replace('None', np.nan).apply(pd.to_numeric, errors='coerce')
        dataset_min_size = min_sizes.get(dataset_name + '_' + data_type, global_min_cluster_size)
        dataset_sizes = [size for size in all_sizes if size >= dataset_min_size]
        if not dataset_sizes:
            print(f"No valid sizes for {dataset_name} ({data_type})")
            continue
        datasets_per_threshold = [
            (df[df['total'] == size]['tp_rate'].dropna().astype(float) * 100).tolist()
            if len(df[df['total'] == size]) > 0 else [0]
            for size in all_sizes
        ]
        medians = [np.median(d) if d else np.nan for d in datasets_per_threshold]
        valid_medians = [m for m in medians if not np.isnan(m) and m > 0]
        valid_positions = [j + 1 for j, d in enumerate(datasets_per_threshold) if d and np.median(d) > 0]
        dataset_min_index = all_sizes.index(dataset_min_size) + 1 if dataset_min_size in all_sizes else 1
        valid_positions = [p for p in valid_positions if p >= dataset_min_index]
        valid_medians = [medians[p-1] for p in valid_positions]

        if valid_medians:
            # Adjust positions for mirrored plot
            plot_positions = [-(p - first_tick_pos) for p in valid_positions] if data_type == 'train' else [p - first_tick_pos for p in valid_positions]
            cumulative_avg = np.cumsum(valid_medians) / np.arange(1, len(valid_medians) + 1)
            line, = ax.plot(plot_positions[:len(cumulative_avg)], cumulative_avg,
                           linewidth=2, color=color)
            avg_value = np.mean(cumulative_avg)
            ax.axhline(y=avg_value, color=color, linestyle=':', linewidth=1.5)
            if dataset_name not in plotted_labels:
                handles.append(line)
                labels.append(dataset_name)
                plotted_labels.add(dataset_name)

    # Configure axes
    ax.spines['left'].set_position(('axes', 0))  # Move y-axis to left edge
    ax.spines['right'].set_color('none')
    ax.spines['bottom'].set_position(('data', 50))
    ax.spines['top'].set_color('none')

    # Set symmetric x-axis limits
    max_pos = max([abs(p) for p in adjusted_positions]) + 1
    ax.set_xlim(-max_pos, max_pos)
    ax_test.set_xlim(max_pos, -max_pos)  # Reverse for mirroring

    # Set x-axis ticks and labels
    ax.set_xticks([-p for p in adjusted_positions[1:]] + [0] + adjusted_positions[1:])
    ax.set_xticklabels([str(s) for s in adjusted_sizes[1:]] + [str(adjusted_sizes[0])] + [str(s) for s in adjusted_sizes[1:]], **TICK_FONT)
    ax_test.set_xticks([])  # Hide test x-axis ticks

    # Label axes
    ax.set_ylabel('True Positive (%)', **TICK_FONT)
    ax.set_xlabel(' <-- Train             Cluster Size            Test -->', **TICK_FONT, x=0.5)
    ax.set_ylim(50, 100)

    # Add legend
    ax.legend(
        handles, labels,
        loc='upper center',
        bbox_to_anchor=(0.5, .15),
        ncol=2,  # Adjusted for fewer datasets
        prop={'size': 22, 'family': 'DejaVu Sans', 'weight': 'bold'},
        frameon=False
    )

    # Add grid
    ax.grid(True, linestyle='--', alpha=0.5)

    # Save plot
    fig.subplots_adjust(top=0.85, left=0.15)
    fig.savefig(output_file, format='pdf', bbox_inches='tight')
    plt.close(fig)

# Example usage (replace with your actual DataFrames)
try:
    plot_cluster_std_dev_mirrored(
        AutoPYara_Train_s3, AutoyaraBase_Train_s3,
        autopyar_s3, autoyar_s3,
        output_file='mirrored_sdhash_0.75.pdf'
    )
except NameError as e:
    print(f"Error: {e}. Please ensure the DataFrames (AutoPYara_Train_s3, AutoyaraBase_Train_s3, autopyar_s3, autoyar_s3) are defined.")